# **Banco de Dados - Não Relacional**

**Instituição:** Pontifícia Universidade Católica de Campinas

**Curso:** Ciência de Dados e Inteligência Artificial

**Professor:** Felipe Cavalaro  

---

## Integrantes

| Nome | RA |
| :--- | :--- |
| Alice Pasolini | 25012938 |
| Enzo Guerra | 25007153 |
| Guilherme Cintra | 25004996 |
| Julia Leandro | 25009148 |
| Lavínia Oliveira | 25894981 |

### **ODS 4 - Educação e Qualidade**

### Indicadores Escolhidos

| Indicador | Fonte do Dataset |
| :--- | :--- |
| Proficiência do SARESP por município | [Acessar dados](https://dados.educacao.sp.gov.br/dataset/profici%C3%AAncia-do-sistema-de-avalia%C3%A7%C3%A3o-de-rendimento-escolar-do-estado-de-s%C3%A3o-paulo-saresp-4) |
| Fluxo escolar por município | [Acessar dados](https://dados.educacao.sp.gov.br/dataset/fluxo-escolar-por-munic%C3%ADpio) |
| Ausências por servidor (Faltas e absenteísmo) | [Acessar dados](https://dados.educacao.sp.gov.br/dataset/aus%C3%AAncias-por-servidor) |
| Censo Escolar | [Acessar dados](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-escolar) |


---
## **🖥️ | IMPLEMENTAÇÃO: ELT** (Extract, Load, Transform)

### **Estrutura Base**
- Bibliotecas e pacotes
- Funções
- Variáveis globais

In [13]:
        # !git clone https://github.com/leasju/pi_educacao_qualidade -q
'''%cd pi_educacao_qualidade/
!pip install -r requirements.txt -q
!npm install localtunnel -q'''

'%cd pi_educacao_qualidade/\n!pip install -r requirements.txt -q\n!npm install localtunnel -q'

Importação das dependências

In [14]:
# DataFrames e funções auxiliares
import pandas as pd
import numpy as np
import time
import os
import zipfile

# Formatar encoding dos arquivos
import unicodedata

## Streamlit Dashboard

In [15]:
import streamlit

In [16]:
'''!curl ipv4.icanhazip.com'''

'!curl ipv4.icanhazip.com'

In [17]:
'''!streamlit run app.py &>/dev/null&
!npx localtunnel --port 8501'''

'!streamlit run app.py &>/dev/null&\n!npx localtunnel --port 8501'

### Funções e estruturas genéricas para a leitura e manipulação dos Datasets

URL Base - GitHub

In [18]:
import os
base_dir = os.getcwd()
if not os.path.exists(os.path.join(base_dir, 'Proficiencia do SARESP')):
    base_dir = os.path.abspath(os.path.join(base_dir, '..'))

caminhoSaresp = os.path.join(base_dir, 'Proficiencia do SARESP') + '/'
caminhoFluxo = os.path.join(base_dir, 'Fluxo Escolar') + '/'
caminhoAusencia = os.path.join(base_dir, 'Ausencia') + '/'
caminhoCenso = os.path.join(base_dir, 'Censo Escolar', 'Censo Escolar') + '/'


Configuração dos anos nos Datasets

In [19]:
# Dicionário com os anos
dict_ano = {
    22: 2022,
    23: 2023,
    24: 2024
}

In [20]:
# Adiciona a coluna 'Ano' aos datasets
def add_col(df, ano):
    df["Ano"] = ano
    return df

Função que realiza a leitura dos Datasets e os concatena ao final

In [21]:
def le_dataset(caminho, arquivo):
    lista_dfs = []

    for k, v in arquivo.items():

        # Detecta formato do dicionário
        # ano : {"nome_arquivo"}
        if isinstance(k, int):
            ano = k
            nome = v
        # "nome_arquivo" : ano
        else:
            nome = k
            ano = v

        url = os.path.join(caminho, nome)

        df = None

        for enc in ['utf-8-sig', 'latin-1']:
            try:
                # print(f"Tentando: {url}")
                df = pd.read_csv(url, sep=';', encoding=enc)
                # print(f"OK: {nome}")
                break
            except Exception as e:
                # print(f"ERRO: {nome} | {e}")
                continue

        if df is not None:
            for chave in dict_ano:
                if str(chave) in nome:
                    df = add_col(df, dict_ano[chave])
                    break
            df.columns = df.columns.str.upper()
            lista_dfs.append(df)

    return pd.concat(lista_dfs, ignore_index=True)

 Função auxiliar para converter strings com vírgula para float (Padronizar)

In [22]:
def padronizar(df, columns):
    for col in columns:
        if col in df.columns:
            # Converte para string, remove espaços, troca vírgula por ponto e converte para float
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.').str.strip(), errors='coerce')
    return df

Lista da região metropolitana de Campinas (10 principais municípios)

In [23]:
regiaoCampinas = ['CAMPINAS','HORTOLANDIA','SUMARE','INDAIATUBA','PAULINIA','AMERICANA','VALINHOS','SANTA BARBARA D\'OESTE']

### ⚪ **DATASET | SARESP**




Leitura e análise do dataframe



In [24]:
sarespLinks = {
    2022: "Proficiencia do SARESP por município de 2022_0.csv",
    2023: "Proficiencia do SARESP por Municipio de 2023_0.csv",
    2024: "Proficiencia do SARESP por Municipio de 2024_0.csv"
}

sarespDF = le_dataset(caminhoSaresp, sarespLinks)
sarespDF

ValueError: No objects to concatenate

In [ ]:
# Cópia do dataframe original (Caso precise)
sarespDFOriginal = sarespDF.copy()
sarespDFOriginal

,DEPADM,DEPBOL,NOMEDEPBOL,CODRMET,CODMUN,MUN,SERIE_ANO,COD_PER,PERIODO,CO_COMP,DS_COMP,MEDPROF,ANO
0,1,1,Rede Estadual,1,100,SAO PAULO,2º Ano EF,1,MANHÃ,1,LÍNGUA PORTUGUESA,"168,9",2022
1,1,1,Rede Estadual,1,100,SAO PAULO,2º Ano EF,1,MANHÃ,2,MATEMÁTICA,"170,1",2022
2,1,1,Rede Estadual,1,100,SAO PAULO,2º Ano EF,2,TARDE,1,LÍNGUA PORTUGUESA,"165,2",2022
3,1,1,Rede Estadual,1,100,SAO PAULO,2º Ano EF,2,TARDE,2,MATEMÁTICA,"173,6",2022
4,1,1,Rede Estadual,1,100,SAO PAULO,2º Ano EF,9,GERAL,1,LÍNGUA PORTUGUESA,"166,5",2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...
38562,2,2,Rede Municipal,5,794,TRABIJU,5º Ano EF,9,GERAL,2,MATEMÁTICA,"230,1",2024
38563,2,2,Rede Municipal,5,794,TRABIJU,9º Ano EF,1,MANHÃ,1,LÍNGUA PORTUGUESA,"235,8",2024
38564,2,2,Rede Municipal,5,794,TRABIJU,9º Ano EF,1,MANHÃ,2,MATEMÁTICA,"265,4",2024
38565,2,2,Rede Municipal,5,794,TRABIJU,9º Ano EF,9,GERAL,1,LÍNGUA PORTUGUESA,"235,8",2024


Normalização do dataframe

In [ ]:
# Remoção das colunas desnecessárias para análise
apagarCol = ['DEPADM','DEPBOL','CODRMET','COD_PER','CO_COMP']
sarespDF = sarespDFOriginal.drop(columns=apagarCol)

- Padronização

In [ ]:
colMEDPROF = ['MEDPROF']
sarespDF = padronizar(sarespDF, colMEDPROF)

In [ ]:
sarespDF = sarespDF[sarespDF.MUN.isin(regiaoCampinas)]

sarespDF = sarespDF.reset_index(drop=True)

sarespDF = sarespDF.rename(columns={
    'MUN': 'MUNICÍPIOS',
    'DS_COMP': 'COMPETÊNCIA',
    'MEDPROF': 'MÉDIA_PROFICIÊNCIA',
    'NOMEDEPBOL': 'REDE',
})

sarespDF = sarespDF[sarespDF['SERIE_ANO'] != 'EM-3ª série']

In [ ]:
colunasAgrupadas = ['MUNICÍPIOS','ANO','SERIE_ANO','COMPETÊNCIA']
coluna = ['MÉDIA_PROFICIÊNCIA']

sarespDF = sarespDF.groupby(colunasAgrupadas)[coluna].mean().reset_index().round(2)

sarespDF.head(20)

,MUNICÍPIOS,ANO,SERIE_ANO,COMPETÊNCIA,MÉDIA_PROFICIÊNCIA
0,AMERICANA,2022,2º Ano EF,LÍNGUA PORTUGUESA,167.17
1,AMERICANA,2022,2º Ano EF,MATEMÁTICA,166.10
2,AMERICANA,2022,5º Ano EF,CIÊNCIAS,225.87
3,AMERICANA,2022,5º Ano EF,LÍNGUA PORTUGUESA,208.20
4,AMERICANA,2022,5º Ano EF,MATEMÁTICA,224.80
5,AMERICANA,2022,9º Ano EF,CIÊNCIAS,272.13
6,AMERICANA,2022,9º Ano EF,LÍNGUA PORTUGUESA,249.83
7,AMERICANA,2022,9º Ano EF,MATEMÁTICA,255.77
8,AMERICANA,2023,2º Ano EF,LÍNGUA PORTUGUESA,176.32
9,AMERICANA,2023,2º Ano EF,MATEMÁTICA,168.42


Download

In [ ]:
sarespDF.to_csv(os.path.join(base_dir, 'Datasets Tratados', 'sarespDFTratado.csv'), index=False)

### ⚪ **DATASET | FLUXO ESCOLAR DOS ALUNOS**
Taxa de Aprovação e Reprovação dos Alunos

Leitura dos datasets

In [ ]:
fluxoLinks = {
    2022: "Fluxo Escolar 2022 - por municipio.csv",
    2023: "Fluxo Escolar 2023 - por município.csv",
    2024: "Fluxo Escolar 2024 - por municipio.csv"
}

fluxoDF = le_dataset(caminhoFluxo, fluxoLinks)
fluxoDF

,ANO_LETIVO,NM_DIRETORIA,NM_MUNICIPIO,CD_REDE_ENSINO,APR_1,REP_1,ABA_1,APR_2,REP_2,ABA_2,APR_3,REP_3,ABA_3,ANO
0,2022,ADAMANTINA,ADAMANTINA,1,0,0,0,"99,61","0,29","0,1","98,97","0,83","0,2",2022
1,2022,ADAMANTINA,DRACENA,1,0,0,0,"97,72","1,74","0,54","97,75","1,86","0,39",2022
2,2022,ADAMANTINA,FLORA RICA,1,0,0,0,0,0,0,100,0,0,2022
3,2022,ADAMANTINA,FLORIDA PAULISTA,1,0,0,0,100,0,0,100,0,0,2022
4,2022,ADAMANTINA,INUBIA PAULISTA,1,0,0,0,"99,35","0,65",0,100,0,0,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1969,2024,TAQUARITINGA,VISTA ALEGRE DO ALTO,1,0,0,0,100,0,0,100,0,0,2024
1970,2024,JALES,VITORIA BRASIL,1,0,0,0,100,0,0,100,0,0,2024
1971,2024,VOTORANTIM,VOTORANTIM,1,0,0,0,"99,01","0,99",0,"98,23","1,77",0,2024
1972,2024,VOTUPORANGA,VOTUPORANGA,1,0,0,0,"99,97","0,03",0,"99,75","0,25",0,2024


In [ ]:
# Cópia do dataframe original(Caso precise)
fluxoDFOriginal = fluxoDF.copy()
fluxoDFOriginal

,ANO_LETIVO,NM_DIRETORIA,NM_MUNICIPIO,CD_REDE_ENSINO,APR_1,REP_1,ABA_1,APR_2,REP_2,ABA_2,APR_3,REP_3,ABA_3,ANO
0,2022,ADAMANTINA,ADAMANTINA,1,0,0,0,"99,61","0,29","0,1","98,97","0,83","0,2",2022
1,2022,ADAMANTINA,DRACENA,1,0,0,0,"97,72","1,74","0,54","97,75","1,86","0,39",2022
2,2022,ADAMANTINA,FLORA RICA,1,0,0,0,0,0,0,100,0,0,2022
3,2022,ADAMANTINA,FLORIDA PAULISTA,1,0,0,0,100,0,0,100,0,0,2022
4,2022,ADAMANTINA,INUBIA PAULISTA,1,0,0,0,"99,35","0,65",0,100,0,0,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1969,2024,TAQUARITINGA,VISTA ALEGRE DO ALTO,1,0,0,0,100,0,0,100,0,0,2024
1970,2024,JALES,VITORIA BRASIL,1,0,0,0,100,0,0,100,0,0,2024
1971,2024,VOTORANTIM,VOTORANTIM,1,0,0,0,"99,01","0,99",0,"98,23","1,77",0,2024
1972,2024,VOTUPORANGA,VOTUPORANGA,1,0,0,0,"99,97","0,03",0,"99,75","0,25",0,2024


Normalização do dataframe

In [ ]:
count = 0
# Mantemos a coluna 'ANO' (ou 'ANO_REF') fora da lista de exclusão
apagarCol = ['ANO_LETIVO','NM_DIRETORIA','ABA_1','ABA_2']

# Percorre as colunas para adicionar as que possuam o número 3 no final (Ensino médio)
for i in fluxoDFOriginal.columns:
  if fluxoDFOriginal.columns[count][-1] == '3':
    apagarCol.append(i)
  count += 1

# Apaga as colunas da lista apagarCol
fluxoDF = fluxoDFOriginal.drop(columns=apagarCol)

- Renomeação das colunas

In [ ]:
fluxoDF = fluxoDF.rename(columns={
    'NM_MUNICIPIO': 'MUNICÍPIOS',
    'APR_1': 'APROVAÇÃO ANOS INICIAIS 9 ANOS',
    'APR_2': 'APROVAÇÃO ANOS FINAIS 9 ANOS',
    'REP_1': 'REPROVAÇÃO ANOS INICIAIS 9 ANOS',
    'REP_2': 'REPROVAÇÃO ANOS FINAIS 9 ANOS'
})
fluxoDF.columns

Index(['MUNICÍPIOS', 'APROVAÇÃO ANOS INICIAIS 9 ANOS',
       'REPROVAÇÃO ANOS INICIAIS 9 ANOS', 'APROVAÇÃO ANOS FINAIS 9 ANOS',
       'REPROVAÇÃO ANOS FINAIS 9 ANOS', 'ANO'],
      dtype='object')

- Padronização

In [ ]:
colFluxo = ['APROVAÇÃO ANOS INICIAIS 9 ANOS','REPROVAÇÃO ANOS INICIAIS 9 ANOS','APROVAÇÃO ANOS FINAIS 9 ANOS','REPROVAÇÃO ANOS FINAIS 9 ANOS']
fluxoDF = padronizar(fluxoDF, colFluxo)

- Transformação das colunas e exibição apenas das necessárias para análise

In [ ]:
# Filtrando pela região de Campinas
fluxoDF = fluxoDF[fluxoDF.MUNICÍPIOS.isin(regiaoCampinas)]

fluxoDF = fluxoDF.reset_index(drop=True)

colunasAgrupadas = ['MUNICÍPIOS', 'CD_REDE_ENSINO', 'ANO']

fluxoDF = fluxoDF.groupby(colunasAgrupadas)[colFluxo].mean().reset_index().round(2)
fluxoDF

,MUNICÍPIOS,ANO,APROVAÇÃO ANOS INICIAIS 9 ANOS,REPROVAÇÃO ANOS INICIAIS 9 ANOS,APROVAÇÃO ANOS FINAIS 9 ANOS,REPROVAÇÃO ANOS FINAIS 9 ANOS
0,AMERICANA,2022,99.55,0.36,97.13,2.23
1,AMERICANA,2023,99.75,0.25,97.38,2.62
2,AMERICANA,2024,99.84,0.16,98.26,1.74
3,CAMPINAS,2022,99.40,0.42,96.90,2.40
4,CAMPINAS,2023,99.39,0.51,97.06,2.63
5,CAMPINAS,2024,99.48,0.52,97.38,2.62
6,HORTOLANDIA,2022,100.00,0.00,98.16,1.19
7,HORTOLANDIA,2023,100.00,0.00,98.42,1.51
8,HORTOLANDIA,2024,100.00,0.00,98.61,1.39
9,INDAIATUBA,2022,0.00,0.00,98.02,1.16


Download

In [ ]:
fluxoDF.to_csv(os.path.join(base_dir, 'Datasets Tratados', 'fluxoDFTratado.csv'), index=False)

### ⚪ **DATASET | AUSÊNCIAS DOS PROFESSORES**

Leitura dos datasets

In [ ]:
ausLinks = {
    f"[dbo].[BASE_AUSENCIAS_{mes:02d}{ano}].csv": 2000 + ano
    for ano in [22, 23, 24]
    for mes in range(1, 13)
}

ausDF = le_dataset(caminhoAusencia, ausLinks)
ausDF

,REGIAO_EXERC,DE_EXERC,CIE_ESCOLA,UA_EXERC,NOME_UA_EXERC,MUNICIPIO_EXERC,QUADRO_EXERC,DI,CARGO_EXERC,NOME_CARGO_EXERC,...,TT_DIAS_FALTA_INJUST,TT_DIAS_LIC_PREMIO,TT_DIAS_LIC_GESTANTE,TT_DIAS_LIC_ACID_TRAB,TT_DIAS_LIC_INTER_PARTIC,TOT_DIAS_AUSENCIAS,TOTAL_DIAS_MES,ID_INTERNO,ANO,TT_DIAS_LIC_PATERNIDADE
0,INTERIOR,D.E.REG. PIRAJU,34459,44966,EE ATALIBA LEONEL,PIRAJU,QM-DOCENTE,1,6407,PROFESSOR EDUCACAO BASICA I,...,0,0,0,0,0,1,31,430958,2022,NaN
1,INTERIOR,D.E.REG. PIRAJU,497824,97181,CEEJA DE PIRAJU,PIRAJU,QM-DOCENTE,1,6409,PROFESSOR EDUCACAO BASICA II,...,0,0,0,0,0,8,31,179714,2022,NaN
2,INTERIOR,D.E.REG. PIRASSUNUNGA,0,39673,D.E.REG. PIRASSUNUNGA,PIRASSUNUNGA,QSE,1,3912,AUXILIAR SERV.GERAIS,...,0,15,0,0,0,15,31,285459,2022,NaN
3,INTERIOR,D.E.REG. PIRASSUNUNGA,0,39673,D.E.REG. PIRASSUNUNGA,PIRASSUNUNGA,QSE,1,4349,OFICIAL ADMINISTRATIVO,...,0,0,0,0,0,31,31,217932,2022,NaN
4,INTERIOR,D.E.REG. PIRASSUNUNGA,19902,43071,EE CESAR LACERDA VERGUEIRO-SEN,ARARAS,QAE,1,4341,AGENTE DE ORGANIZACAO ESCOLAR,...,0,0,0,0,0,12,31,195380,2022,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2213687,CAPITAL,D.E.REG. SUL 2,36298,45962,EE JOAO SUSSUMU HIRATA DEP.,SAO PAULO,QM-DOCENTE,2,5774,PROFESSOR DE ENSINO FUNDAMENTAL E MEDIO,...,0,0,0,0,0,1,31,405087,2024,0.0
2213688,CAPITAL,D.E.REG. SUL 2,36705,45958,EE HONORIO MONTEIRO DR.,SAO PAULO,QM-DOCENTE,1,5774,PROFESSOR DE ENSINO FUNDAMENTAL E MEDIO,...,0,0,0,0,0,1,31,501697,2024,0.0
2213689,CAPITAL,D.E.REG. SUL 2,36705,45958,EE HONORIO MONTEIRO DR.,SAO PAULO,QM-DOCENTE,1,5774,PROFESSOR DE ENSINO FUNDAMENTAL E MEDIO,...,0,0,0,0,0,1,31,424925,2024,0.0
2213690,CAPITAL,D.E.REG. SUL 2,36705,45958,EE HONORIO MONTEIRO DR.,SAO PAULO,QM-DOCENTE,1,6407,PROFESSOR EDUCACAO BASICA I,...,0,0,0,0,0,19,31,281964,2024,0.0


In [ ]:
# Cópia do dataframe original (Caso precise)
ausDFOriginal = ausDF.copy()
ausDFOriginal

,REGIAO_EXERC,DE_EXERC,CIE_ESCOLA,UA_EXERC,NOME_UA_EXERC,MUNICIPIO_EXERC,QUADRO_EXERC,DI,CARGO_EXERC,NOME_CARGO_EXERC,...,TT_DIAS_FALTA_INJUST,TT_DIAS_LIC_PREMIO,TT_DIAS_LIC_GESTANTE,TT_DIAS_LIC_ACID_TRAB,TT_DIAS_LIC_INTER_PARTIC,TOT_DIAS_AUSENCIAS,TOTAL_DIAS_MES,ID_INTERNO,ANO,TT_DIAS_LIC_PATERNIDADE
0,INTERIOR,D.E.REG. PIRAJU,34459,44966,EE ATALIBA LEONEL,PIRAJU,QM-DOCENTE,1,6407,PROFESSOR EDUCACAO BASICA I,...,0,0,0,0,0,1,31,430958,2022,NaN
1,INTERIOR,D.E.REG. PIRAJU,497824,97181,CEEJA DE PIRAJU,PIRAJU,QM-DOCENTE,1,6409,PROFESSOR EDUCACAO BASICA II,...,0,0,0,0,0,8,31,179714,2022,NaN
2,INTERIOR,D.E.REG. PIRASSUNUNGA,0,39673,D.E.REG. PIRASSUNUNGA,PIRASSUNUNGA,QSE,1,3912,AUXILIAR SERV.GERAIS,...,0,15,0,0,0,15,31,285459,2022,NaN
3,INTERIOR,D.E.REG. PIRASSUNUNGA,0,39673,D.E.REG. PIRASSUNUNGA,PIRASSUNUNGA,QSE,1,4349,OFICIAL ADMINISTRATIVO,...,0,0,0,0,0,31,31,217932,2022,NaN
4,INTERIOR,D.E.REG. PIRASSUNUNGA,19902,43071,EE CESAR LACERDA VERGUEIRO-SEN,ARARAS,QAE,1,4341,AGENTE DE ORGANIZACAO ESCOLAR,...,0,0,0,0,0,12,31,195380,2022,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2213687,CAPITAL,D.E.REG. SUL 2,36298,45962,EE JOAO SUSSUMU HIRATA DEP.,SAO PAULO,QM-DOCENTE,2,5774,PROFESSOR DE ENSINO FUNDAMENTAL E MEDIO,...,0,0,0,0,0,1,31,405087,2024,0.0
2213688,CAPITAL,D.E.REG. SUL 2,36705,45958,EE HONORIO MONTEIRO DR.,SAO PAULO,QM-DOCENTE,1,5774,PROFESSOR DE ENSINO FUNDAMENTAL E MEDIO,...,0,0,0,0,0,1,31,501697,2024,0.0
2213689,CAPITAL,D.E.REG. SUL 2,36705,45958,EE HONORIO MONTEIRO DR.,SAO PAULO,QM-DOCENTE,1,5774,PROFESSOR DE ENSINO FUNDAMENTAL E MEDIO,...,0,0,0,0,0,1,31,424925,2024,0.0
2213690,CAPITAL,D.E.REG. SUL 2,36705,45958,EE HONORIO MONTEIRO DR.,SAO PAULO,QM-DOCENTE,1,6407,PROFESSOR EDUCACAO BASICA I,...,0,0,0,0,0,19,31,281964,2024,0.0


Normalização do dataframe

In [ ]:
# Apaga as colunas dentro da lista apagarCol
apagarCol = ['REGIAO_EXERC', 'DE_EXERC', 'NOME_UA_EXERC', 'NOME_CARGO_EXERC', 'CIE_ESCOLA', 'UA_EXERC', 'DI', 'CARGO_EXERC', 'CATEG_E']
ausDF = ausDFOriginal.drop(columns=apagarCol)

# Normaliza todos os municípios dentro da coluna para que não haja espaços vazios e todos sejam maiúsculos
ausDF['MUNICIPIO_EXERC'] = ausDF['MUNICIPIO_EXERC'].str.strip().str.upper()

# Seleciona apenas os municípios da região de Campinas
ausDF = ausDF[ausDF['MUNICIPIO_EXERC'].isin(regiaoCampinas)]

- Renomeação das colunas

In [ ]:
ausDF = ausDF.rename(columns={
    'MUNICIPIO_EXERC': 'MUNICÍPIOS',
    'TT_DIAS_FALTA_JUST': 'TOTAL FALTAS JUST',
    'TT_DIAS_FALTA_INJUST': 'TOTAL FALTAS INJUST',
})

In [ ]:
ausDF['TOTAL DIAS AUSENTES'] = (ausDF['TOTAL FALTAS JUST'] + ausDF['TOTAL FALTAS INJUST'])

# Filtrando o total de faltas por ano
ausDF = (ausDF.groupby(['MUNICÍPIOS','ANO'])[['TOTAL FALTAS JUST','TOTAL FALTAS INJUST', 'TOTAL DIAS AUSENTES']].sum().reset_index())

ausDF

,MUNICÍPIOS,ANO,TOTAL FALTAS JUST,TOTAL FALTAS INJUST,TOTAL DIAS AUSENTES
0,AMERICANA,2022,1363,488,1851
1,AMERICANA,2023,772,223,995
2,AMERICANA,2024,490,84,574
3,CAMPINAS,2022,8637,4168,12805
4,CAMPINAS,2023,6328,3824,10152
5,CAMPINAS,2024,3999,2974,6973
6,HORTOLANDIA,2022,1175,495,1670
7,HORTOLANDIA,2023,1067,847,1914
8,HORTOLANDIA,2024,683,404,1087
9,INDAIATUBA,2022,1394,1870,3264


Download

In [ ]:
ausDF.to_csv(os.path.join(base_dir, 'Datasets Tratados', 'ausDFTratado.csv'), index=False)

### ⚪ **DATASET CENSO ESCOLAR**

In [ ]:
# Caminho do arquivo zip e diretório de destino
arquivo_zip = '../Censo Escolar/Censo_zipado.zip'
pasta_destino = '../Censo Escolar'

# Abre o arquivo zip e extrai o conteúdo
with zipfile.ZipFile(arquivo_zip, 'r') as zip_ref:
    zip_ref.extractall(pasta_destino)
    print(f"Arquivos extraídos em: {pasta_destino}")


Arquivos extraídos em: /content/pi_educacao_qualidade/Censo Escolar


In [ ]:
links_censo = {
    2022: "microdados_ed_basica_2022.csv",
    2023: "microdados_ed_basica_2023.csv",
    2024: "microdados_ed_basica_2024.csv",
}

censoDF = le_dataset(caminhoCenso, links_censo)
censoDF

/tmp/ipykernel_22061/240922229.py:23: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url, sep=';', encoding=enc)
/tmp/ipykernel_22061/240922229.py:23: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url, sep=';', encoding=enc)
/tmp/ipykernel_22061/240922229.py:23: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url, sep=';', encoding=enc)


,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_MESORREGIAO,CO_MESORREGIAO,...,IN_MATERIAL_PED_AGRICOLA,IN_MATERIAL_PED_QUILOMBOLA,IN_MATERIAL_PED_EDU_ESP,IN_EDUC_AMBIENTAL,IN_EDUC_AMB_CONTEUDO,IN_EDUC_AMB_CURRICULAR,IN_EDUC_AMB_EIXO,IN_EDUC_AMB_EVENTOS,IN_EDUC_AMB_PROJETOS,IN_EDUC_AMB_NENHUMA
0,2022,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Leste Rondoniense,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Leste Rondoniense,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Leste Rondoniense,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Leste Rondoniense,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Leste Rondoniense,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
657814,2024,Centro-Oeste,5,Distrito Federal,DF,53,Brasília,5300108,Distrito Federal,1,...,0.0,0.0,0.0,0.0,9.0,9.0,9.0,9.0,9.0,9.0
657815,2024,Centro-Oeste,5,Distrito Federal,DF,53,Brasília,5300108,Distrito Federal,1,...,0.0,0.0,0.0,0.0,9.0,9.0,9.0,9.0,9.0,9.0
657816,2024,Centro-Oeste,5,Distrito Federal,DF,53,Brasília,5300108,Distrito Federal,1,...,0.0,0.0,0.0,0.0,9.0,9.0,9.0,9.0,9.0,9.0
657817,2024,Centro-Oeste,5,Distrito Federal,DF,53,Brasília,5300108,Distrito Federal,1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
# Cópia do dataframe original(Caso precise)
censoDFOriginal = censoDF.copy()
censoDFOriginal

,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_MESORREGIAO,CO_MESORREGIAO,...,IN_MATERIAL_PED_AGRICOLA,IN_MATERIAL_PED_QUILOMBOLA,IN_MATERIAL_PED_EDU_ESP,IN_EDUC_AMBIENTAL,IN_EDUC_AMB_CONTEUDO,IN_EDUC_AMB_CURRICULAR,IN_EDUC_AMB_EIXO,IN_EDUC_AMB_EVENTOS,IN_EDUC_AMB_PROJETOS,IN_EDUC_AMB_NENHUMA
0,2022,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Leste Rondoniense,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Leste Rondoniense,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Leste Rondoniense,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Leste Rondoniense,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Leste Rondoniense,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
657814,2024,Centro-Oeste,5,Distrito Federal,DF,53,Brasília,5300108,Distrito Federal,1,...,0.0,0.0,0.0,0.0,9.0,9.0,9.0,9.0,9.0,9.0
657815,2024,Centro-Oeste,5,Distrito Federal,DF,53,Brasília,5300108,Distrito Federal,1,...,0.0,0.0,0.0,0.0,9.0,9.0,9.0,9.0,9.0,9.0
657816,2024,Centro-Oeste,5,Distrito Federal,DF,53,Brasília,5300108,Distrito Federal,1,...,0.0,0.0,0.0,0.0,9.0,9.0,9.0,9.0,9.0,9.0
657817,2024,Centro-Oeste,5,Distrito Federal,DF,53,Brasília,5300108,Distrito Federal,1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


- Padronização

In [ ]:
censoDF['NO_MUNICIPIO'] = censoDFOriginal['NO_MUNICIPIO'].str.strip().str.upper()
# Removido filtro aqui, faremos APOS a normalizacao dos acentos!

def normalizar(texto):
    return unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii').upper().strip()

censoDF['NO_MUNICIPIO'] = censoDF['NO_MUNICIPIO'].apply(normalizar)
censoDF = censoDF[censoDF['NO_MUNICIPIO'].isin(regiaoCampinas)].reset_index(drop=True)

censoDF = censoDF.rename(columns={'NO_MUNICIPIO': 'MUNICÍPIOS'})

# Agrega por município e ano (soma matrículas e docentes, média de infraestrutura)
censoDF = censoDF.groupby(['MUNICÍPIOS', 'CO_MUNICIPIO', 'ANO', 'TP_DEPENDENCIA']).agg(
  **{

    # BOOLEANAS - Começa com 'IN'
    # Infraestrutura
    'BIBLIOTECA': ('IN_BIBLIOTECA', 'mean'),
    'BIBLIOTECA OU SALA DE LEITURA': ('IN_BIBLIOTECA_SALA_LEITURA', 'mean'),
    'LAB. INFORMÁTICA': ('IN_LABORATORIO_INFORMATICA', 'mean'),
    'LAB. CIÊNCIAS': ('IN_LABORATORIO_CIENCIAS', 'mean'),
    'QUADRA COBERTA': ('IN_QUADRA_ESPORTES_COBERTA', 'mean'),
    'QUADRA DESCOBERTA': ('IN_QUADRA_ESPORTES_DESCOBERTA', 'mean'),
    'REFEITÓRIO': ('IN_REFEITORIO', 'mean'),

    # Infraestrutura básica
    'ÁGUA POTÁVEL': ('IN_AGUA_POTAVEL', 'mean'),
    'ESGOTO': ('IN_ESGOTO_REDE_PUBLICA', 'mean'),
    'ENERGIA ELÉTRICA': ('IN_ENERGIA_REDE_PUBLICA', 'mean'),

    # Tecnologia
    'Quantidade de computadores em uso pelos alunos': ('QT_COMP_ALUNO', 'sum'),
    'INTERNET': ('IN_INTERNET', 'mean'),
    'INTERNET - ALUNOS': ('IN_INTERNET_ALUNOS', 'mean'),
    'INTERNET - BANDA LARGA': ('IN_BANDA_LARGA', 'mean'),

    # NUMÉRICAS --------
    'QTDE LOUSAS DIGITAIS': ('QT_EQUIP_LOUSA_DIGITAL', 'sum'),
    'QTDE DATASHOW': ('QT_EQUIP_MULTIMIDIA', 'sum'),

    # Docentes
    'QTDE DOCENTES FUNDAMENTAL - TOTAL': ('QT_DOC_FUND', 'sum'),
    'QTDE DOCENTES FUNDAMENTAL - ANOS INICIAIS': ('QT_DOC_FUND_AI', 'sum'),
    'QTDE DOCENTES FUNDAMENTAL - ANOS FINAIS': ('QT_DOC_FUND_AF', 'sum'),

    # Funcionários
    'QTDE FUNCIONARIOS ESCOLA': ('QT_FUNCIONARIOS', 'sum'),
    'QTDE PROFISSIONAIS ESCOLA - APOIO E SUPERVISÃO PEDAGÓGICA': ('QT_PROF_PEDAGOGIA', 'sum'),

    # Matrículas
    'QTDE MATRICULAS FUNDAMENTAL - TOTAL': ('QT_MAT_FUND', 'sum'),
    'QTDE MATRICULAS FUNDAMENTAL - ANOS INICIAIS': ('QT_MAT_FUND_AI', 'sum'),
    'QTDE MATRICULAS FUNDAMENTAL - ANOS FINAIS': ('QT_MAT_FUND_AF', 'sum'),

    # Turmas
    'QTDE TURMAS FUNDAMENTAL - TOTAL': ('QT_TUR_FUND', 'sum'),
    'QTDE TURMAS FUNDAMENTAL - ANOS INICIAIS': ('QT_TUR_FUND_AI', 'sum'),
    'QTDE TURMAS FUNDAMENTAL - ANOS FINAIS': ('QT_TUR_FUND_AF', 'sum'),

    # Estrutura
    'QTDE SALAS DE AULA ESCOLA': ('QT_SALAS_EXISTENTES', 'sum'),

    # Cor/Raça
    'QTDE MATRICULAS FUNDAMENTAL - Cor/Raça Branca': ('QT_MAT_BAS_BRANCA', 'sum'),
    'QTDE MATRICULAS FUNDAMENTAL - Cor/Raça Preta': ('QT_MAT_BAS_PRETA', 'sum'),
    'QTDE MATRICULAS FUNDAMENTAL - Cor/Raça Parda': ('QT_MAT_BAS_PARDA', 'sum'),
    'QTDE MATRICULAS FUNDAMENTAL - Cor/Raça Indígena': ('QT_MAT_BAS_INDIGENA', 'sum'),

    # Turno
    'QTDE MATRICULAS FUNDAMENTAL - Turno Diurno': ('QT_MAT_BAS_D', 'sum'),
    'QTDE MATRICULAS FUNDAMENTAL - Turno Noturno': ('QT_MAT_BAS_N', 'sum'),

    # Categoria
    'LOCALIZAÇÃO': ('TP_LOCALIZACAO', 'first'),

    # Etapas de ensino
    'ETAPAS ENSINO - FUNDAMENTAL - TOTAL  (Possui uma ou mais matrículas)': ('IN_FUND', 'mean'),
    'ETAPAS ENSINO - FUNDAMENTAL - ANOS INICIAIS (Possui uma ou mais matrículas)': ('IN_FUND_AI', 'mean'),
    'ETAPAS ENSINO - FUNDAMENTAL - ANOS FINAIS (Possui uma ou mais matrículas)': ('IN_FUND_AF', 'mean'),

  }
).reset_index().round(2)

ValueError: Grouper for 'MUNICÍPIOS' not 1-dimensional

In [ ]:
censoDF.columns

Index(['MUNICÍPIOS', 'ANO', 'BIBLIOTECA', 'BIBLIOTECA OU SALA DE LEITURA',
       'LAB. INFORMÁTICA', 'LAB. CIÊNCIAS', 'QUADRA COBERTA',
       'QUADRA DESCOBERTA', 'REFEITÓRIO', 'BANHEIRO DENTRO DO PRÉDIO',
       'ÁGUA POTÁVEL', 'ESGOTO', 'ENERGIA ELÉTRICA',
       'Quantidade de computadores em uso pelos alunos', 'INTERNET',
       'INTERNET - ALUNOS', 'INTERNET - BANDA LARGA', 'QTDE LOUSAS DIGITAIS',
       'QTDE DATASHOW', 'QTDE DOCENTES FUNDAMENTAL - TOTAL',
       'QTDE DOCENTES FUNDAMENTAL - ANOS INICIAIS',
       'QTDE DOCENTES FUNDAMENTAL - ANOS FINAIS', 'QTDE FUNCIONARIOS ESCOLA',
       'QTDE PROFISSIONAIS ESCOLA - APOIO E SUPERVISÃO PEDAGÓGICA',
       'QTDE MATRICULAS FUNDAMENTAL - TOTAL',
       'QTDE MATRICULAS FUNDAMENTAL - ANOS INICIAIS',
       'QTDE MATRICULAS FUNDAMENTAL - ANOS FINAIS',
       'QTDE TURMAS FUNDAMENTAL - TOTAL',
       'QTDE TURMAS FUNDAMENTAL - ANOS INICIAIS',
       'QTDE TURMAS FUNDAMENTAL - ANOS FINAIS', 'QTDE SALAS DE AULA ESCOLA',
       'QTDE

In [ ]:
censoDF.tail()

,MUNICÍPIOS,ANO,BIBLIOTECA,BIBLIOTECA OU SALA DE LEITURA,LAB. INFORMÁTICA,LAB. CIÊNCIAS,QUADRA COBERTA,QUADRA DESCOBERTA,REFEITÓRIO,BANHEIRO DENTRO DO PRÉDIO,...,QTDE MATRICULAS FUNDAMENTAL - Cor/Raça Preta,QTDE MATRICULAS FUNDAMENTAL - Cor/Raça Parda,QTDE MATRICULAS FUNDAMENTAL - Cor/Raça Indígena,QTDE MATRICULAS FUNDAMENTAL - Turno Diurno,QTDE MATRICULAS FUNDAMENTAL - Turno Noturno,Dependência Administrativa,LOCALIZAÇÃO,ETAPAS ENSINO - FUNDAMENTAL - TOTAL (Possui uma ou mais matrículas),ETAPAS ENSINO - FUNDAMENTAL - ANOS INICIAIS (Possui uma ou mais matrículas),ETAPAS ENSINO - FUNDAMENTAL - ANOS FINAIS (Possui uma ou mais matrículas)
7,INDAIATUBA,2023,0.17,0.57,0.41,0.11,0.39,0.13,0.64,NaN,...,852.0,12534.0,50.0,54646.0,4013.0,3,1,0.51,0.37,0.25
8,INDAIATUBA,2024,0.19,0.59,0.40,0.13,0.40,0.15,0.66,NaN,...,1109.0,14330.0,64.0,57916.0,4398.0,3,1,0.51,0.37,0.26
9,VALINHOS,2022,0.13,0.48,0.38,0.10,0.36,0.22,0.66,NaN,...,320.0,3471.0,10.0,23630.0,1381.0,4,1,0.72,0.65,0.34
10,VALINHOS,2023,0.20,0.51,0.36,0.10,0.37,0.21,0.70,NaN,...,358.0,3692.0,16.0,23768.0,1380.0,4,1,0.67,0.62,0.34
11,VALINHOS,2024,0.20,0.51,0.34,0.09,0.39,0.18,0.74,NaN,...,372.0,3744.0,15.0,23831.0,1315.0,4,1,0.61,0.53,0.33


Download

In [ ]:
censoDF.to_csv(os.path.join(base_dir, 'Datasets Tratados', 'censoDFTratado.csv'), index=False)